In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import joblib
import re
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import sys

from plotly.subplots import make_subplots

from sklearn.feature_extraction.text import CountVectorizer   # Sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer    # TF-IDF
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS  #Stop words English
)

import nltk
nltk.download('wordnet', download_dir="./nltk_data")
nltk.data.path.append('./nltk_data')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer   # Lemmatisation
from nltk.stem.snowball import SnowballStemmer

from contraction_fix import fix as expand_contractions # Contraction

import spacy                                # NLP avancé
from spacy import displacy                  # Visualisation
!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package wordnet to ./nltk_data...


In [ ]:
df=pd.read_csv('scitweets_export.tsv',
sep='\t')
print (df.head())
print (df.shape)
print (df.columns)

In [ ]:
emoticons_str = r"""
    (?:
        [:=;] # Eyes
        [oO\-]? # Nose (optional)
        [D\)\]\(\]/\\OpP] # Mouth
    )"""

#Prise en compte des éléments qui doivent être regroupés
regex_str = [
    emoticons_str,
    r'<[^>]+>', # HTML tags
    r'(?:@[\w_]+)', # @-mentions
    r"(?:\#+[\w_]+[\w\'_\-]*[\w_]+)", # hash-tags
    r'http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+', # URLs

    r'(?:(?:\d+,?)+(?:\.?\d+)?)', # nombres
    r"(?:[a-z][a-z'\-_]+[a-z])", # mots avec - et '
    r'(?:[\w_]+)', # autres mots
    r'(?:\S)' # le reste
]

tokens_re = re.compile(r'('+'|'.join(regex_str)+')', re.VERBOSE | re.IGNORECASE)
emoticon_re = re.compile(r'^'+emoticons_str+'$', re.VERBOSE | re.IGNORECASE)

def tokenize(s):
    return tokens_re.findall(s)

def preprocess(s, emoticon=True, lowercase=False):
    tokens = tokenize(s)
    if lowercase:
        tokens = [tok.lower() for tok in tokens]
    if not emoticon:
        tokens = [tok for tok in tokens if not emoticon_re.search(tok)]
    return tokens

In [ ]:

print(tokenize(df.text[618]))
print(preprocess(df.text[618], False, True))

In [ ]:
def stop_words(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        if key not in ENGLISH_STOP_WORDS:
            expanded.append(t)
    return " ".join(expanded)

text = "I’m happy but I don't know why."
print(text)
print(stop_words(expand_contractions(text)))

In [ ]:
stemmer = nltk.stem.porter.PorterStemmer()#PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stemmerLemma(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    expanded = []
    t2 = nlp(text)
    
    for t in t2:
        
        key = t.text.lower()
        print(key, t.pos_, stemmer.stem(key), lemmatizer.lemmatize(key))
        if t.pos_ == "VERB":
            result = stemmer.stem(key)
        elif t.pos_ == "NOUN" or t.pos_ == "ADJ" or t.pos_ == "PROPN":
            result = lemmatizer.lemmatize(key)
        else:
            result = key
        expanded.append(result)
    return " ".join(expanded)
print(df.text[3])
print(stemmerLemma(df.text[3]))
text = "I want this cake"
print(stop_words(stemmerLemma(text)))
print(stemmerLemma("I am running"))